In [86]:
import numpy as np
import math
import random

import json
import scipy
from scipy.optimize import minimize
from scipy.optimize import fsolve
from scipy.stats import norm
import tqdm
import torch
import matplotlib.pyplot as plt

In [87]:

# Define constants
TRUE_COEFF_SECOND = np.array([0,1.1,-0.5])
TRUE_COEFF_FOURTH = np.array([-150, 480, -165, 22, -1])


def func_revenue(price,dim):
    if dim == 4:
        TRUE_COEFF = TRUE_COEFF_FOURTH
    elif dim == 2:
        TRUE_COEFF = TRUE_COEFF_SECOND
    p_arr = np.array([price ** i for i in range(len(TRUE_COEFF))])
    return np.sum(p_arr * TRUE_COEFF)


optimal_reward_second =  0.605
optimal_reward_fourth = 323.608
optimal_reward_exponential =  100

p_min = 1
p_max = 4
var = 10
# initial price
initial_price = []
initial_price_second = []

def compute_barycentre(dim=4):
    """Compute the barycentric spanner."""
    def equations(p):
        p2, p3, p4 = p
        return (
            1/p2 + 1/(p2-p3) + 1/(p2-p4) + 1/(p2-1),
            1/p3 + 1/(p3-p2) + 1/(p3-p4) + 1/(p3-1),
            1/p4 + 1/(p4-p2) + 1/(p4-p3) + 1/(p4-1),
        )

    while True:
        b2_init = np.random.uniform(0, 1)
        b3_init = np.random.uniform(0, 1)
        b4_init = np.random.uniform(0, 1)
        b2, b3, b4 = fsolve(equations, (b2_init, b3_init, b4_init))
        if 0 <= b2 <= 1 and 0 <= b3 <= 1 and 0 <= b4 <= 1:
            break

    if dim == 4:
        return np.array([0., b2, b3, b4, 1.]).reshape(-1, 1)
    elif dim == 2:
        return np.array([0., 0.5, 1.]).reshape(-1, 1)


barycentric_spanner = compute_barycentre(dim=4)
print(barycentric_spanner)

[[0.        ]
 [0.17267316]
 [0.82732684]
 [0.5       ]
 [1.        ]]


In [88]:
def Thompson(actual_dim,model_dim):
    
    if model_dim == 4:
        TRUE_COEFF = TRUE_COEFF_FOURTH
        optimal_reward=optimal_reward_fourth
    elif model_dim == 2:
        TRUE_COEFF = TRUE_COEFF_SECOND
        optimal_reward=optimal_reward_second
    
    epsilon_array = []
    def p_vector(p):
        p_vec = np.array([p**i for i in range(TRUE_COEFF.shape[0])])
        return p_vec.reshape(len(p_vec),1)

    def g_fun(p,*args):
        p = p[0]
        p_vec = np.array([p**i for i in range(TRUE_COEFF.shape[0])])
        coeff_array = args[0]
        return -np.sum(coeff_array*p_vec)
    
    var_cov_inv = np.zeros((TRUE_COEFF.shape[0],TRUE_COEFF.shape[0]))
    barycentric_spanner = compute_barycentre(dim = TRUE_COEFF.shape[0]-1)
    # print(barycentric_spanner)
    for b in barycentric_spanner:
        b_vec = p_vector(b)
        var_cov_inv += b_vec@np.transpose(b_vec)
    
    var_cov = np.linalg.inv(var_cov_inv)
    mu_ = np.array([0]*TRUE_COEFF.shape[0])
    pl = p_min 
    ph = p_max
    w = np.random.multivariate_normal(mu_,var_cov)
    price_estimate = minimize(g_fun,x0 = np.array([random.random()]),args = (w,),bounds = [(pl,ph)])["x"][0]
    initial_price.append(price_estimate)
    initial_price_second.append(price_estimate)
    p_array = []
    p_curr = p_vector(price_estimate)
    p_array.append(price_estimate)
    reven_array = []
    def update_mean(rev,p_ar):
      sum = 0
      for i in range(len(rev)):
          price = p_ar[i]
          p = p_vector(price)
          sum += rev[i]*p/100
      return sum

    def update_mat(p_ar):
        sum = 0
        for i in range(len(p_ar)):
            price = p_ar[i]
            p = p_vector(price)
            sum += (p@np.transpose(p))/100
        return sum
    error = 0
    for i in range(100):
        # rev_curr = (true_coeff.reshape(1,3)@p_curr)[0][0]+np.random.normal(0,10)
        rev_curr = func_revenue(price_estimate,actual_dim)+np.random.normal(0,var)
        
        reven_array.append(rev_curr)
        var_copy = var_cov
        mu_copy = mu_
        var_cov = np.linalg.inv(np.linalg.inv(var_copy) + update_mat(p_array))
        # np.linalg.eig(var_cov)
        min_ev = np.min(np.linalg.eig(var_cov)[0])
        if min_ev <= 0:
            var_cov += np.diag(np.array([1e-5]*TRUE_COEFF.shape[0]))
        mu_ = var_cov@(np.linalg.inv(var_copy)@mu_copy.reshape(TRUE_COEFF.shape[0],1) + update_mean(reven_array,p_array))
    #     mu_ = var_cov@( rev_curr*p_curr/100)
        mu_ = mu_.reshape(TRUE_COEFF.shape[0],)
        w_sample = np.random.multivariate_normal(mu_,var_cov)
        price_estimate = minimize(g_fun,x0 = np.array([random.random()]),args = (w_sample,),bounds = [(pl,ph)])["x"][0]
        p_curr = p_vector(price_estimate)
        p_array.append(price_estimate)
        # print(rev_curr)
        # epsilon = max(0,100-reven_array[-1])
        epsilon =np.max([0,np.min(optimal_reward - np.array(reven_array))])
        # error += epsilon
        epsilon_array.append(epsilon)
    return np.array(epsilon_array),reven_array



In [93]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_model
from botorch.acquisition import ExpectedImprovement
from botorch.optim import optimize_acqf
from gpytorch.mlls.exact_marginal_log_likelihood import ExactMarginalLogLikelihood
from botorch.models.transforms import Normalize, Standardize


def environment_response(mu_true, p, sigma, actual_dim=4):
    return func_revenue(p, actual_dim)+np.random.normal(0, sigma)


def bo_price_optimization(mu_true, sigma_noise, total_iterations, p_min, p_max, n, i, actual_dim=4, model_dim=4):
    # Initialize prices and rewards storage
    prices = []
    rewards = []
    regrets_price = []
    initial_price = initial_price_second
    # initial_price = [1]

    # Initial training data: Randomly sample a few prices to initialize
    # train_x = torch.distributions.Uniform(p_min, p_max).sample((1, 1))
    train_x = torch.tensor(
        [initial_price[i]], dtype=torch.float64).unsqueeze(1)
    # make train_x as float64
    train_x = train_x.type(torch.float64)
    reward = environment_response(
        mu_true, train_x.reshape(-1)[0], sigma_noise, actual_dim)
    print(reward)
    regret = max(0, optimal_reward_fourth - reward)

    regrets_price.append(regret)
    train_y = torch.tensor([reward], dtype=torch.float64).unsqueeze(1)
    for t in range(total_iterations):
        # Step 1: Fit a GP model to the current data
        gp = SingleTaskGP(train_x, train_y, outcome_transform=Standardize(m=1), input_transform=Normalize(
            d=1, bounds=torch.tensor([[p_min], [p_max]], dtype=torch.float64)))
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_model(mll)

        # Step 2: Use Expected Improvement (EI) acquisition function to suggest the next price
        best_f = train_y.max().item()
        EI = ExpectedImprovement(gp, best_f=best_f)
        new_price, _ = optimize_acqf(
            acq_function=EI,
            bounds=torch.tensor([[p_min], [p_max]], dtype=torch.float64),
            q=1,  # Optimize for a single new point
            num_restarts=20,  # Multiple restarts to find the optimal point
            raw_samples=1024  # Number of raw samples for exploration
        )
        prices.append(new_price.item())

        # Step 3: Get the environment response (reward) at the suggested price
        reward = environment_response(
            mu_true, new_price.reshape(-1)[0], sigma_noise)
        rewards.append(reward)
        # print(train_x.shape)
        # print(train_y.shape)
        # Step 4: Update the training set with the new data
        train_x = torch.cat([train_x, new_price])
        train_y = torch.cat(
            [train_y, torch.tensor([[reward]], dtype=torch.float64)])

        regret = max(0, optimal_reward_fourth -
                     max(np.array(train_y.reshape(-1))))
        regrets_price.append(regret)
    print(regrets_price)
    print(train_x)
    print(train_y)
    return regrets_price, rewards


TRUE_COEFF_SECOND = np.array([0, 1.1, -0.5])
TRUE_COEFF_FOURTH = np.array([-150, 480, -165, 22, -1])

# Hyperparameters
mu_true = TRUE_COEFF_FOURTH  # True weight vector
sigma_noise = var  # Noise in the environment
total_iterations = 100  # Number of iterations
p_min, p_max = p_min, p_max  # Range of prices
n = TRUE_COEFF_FOURTH.shape[0]  # Degree of polynomial features

# Run the Bayesian Optimization algorithm
regrets_price, rewards = bo_price_optimization(
    mu_true, sigma_noise, total_iterations, p_min, p_max, n, 0)

bo_regret_Avg = []
reward_Avg = []
for i in range(10):
    regrets_price, rewards = bo_price_optimization(
        mu_true, sigma_noise, total_iterations, p_min, p_max, n, i)
    bo_regret_Avg.append(regrets_price)
    reward_Avg.append(rewards)

IndexError: list index out of range

In [ ]:
def ParameterSpace(i):
    epsilon_array = []
    mu_ = np.zeros(TRUE_COEFF.shape[0])
    # mu_ = np.array([-145,450,-150,10,-0.3])
    sigma_ = np.random.uniform(0, 1, TRUE_COEFF.shape[0])

    def p_vector(p):
        p_arr = np.array([p**i for i in range(TRUE_COEFF.shape[0])])
        return p_arr

    def g_fun(p, *args):
        p = p[0]
        p_vec = np.array(
            [p**i for i in range(TRUE_COEFF.shape[0])]).reshape(-1)
        args = args[0]
        coeff_array = np.array(args).flatten()
        return -np.sum(coeff_array*p_vec)
    w = mu_ + sigma_ * np.random.normal(0, 1)
    pl = float(p_min)
    ph = float(p_max)

    # price_estimate = minimize(g_fun,x0 = np.array([random.random()]),args = (w),bounds = [(pl,ph)])["x"][0]
    price_estimate = initial_price[i]
    p_array = []
    reven_array_true = []
    p_curr = p_vector(price_estimate)
    p_array.append(price_estimate)
    alpha = 0.01
    error = 0
    for t in range(1, 101):
        # rev_curr = (true_coeff.reshape(1,len(true_coeff))@p_curr.reshape(len(p_curr),1))[0][0]+np.random.normal(0,10)
        rev_curr = func_revenue(price_estimate)+np.random.normal(0, var)
    #     rev_curr_predict = (w.reshape(1,5)@p_curr)[0][0]
    #     print(rev_curr)
        reven_array_true.append(rev_curr)
    #     reven_array_predicted.append(rev_curr_predict
        sigma_ = sigma_*np.random.normal(0, 1)
        mu = torch.tensor(mu_.reshape(len(mu_), 1), requires_grad=True)
        sigma = torch.tensor(sigma_.reshape(
            len(sigma_), 1), requires_grad=True)
        sum = torch.tensor([0])
        for i in range(len(p_array)):
            p_vec = p_vector(p_array[i])
            p_vec = p_vec.reshape(len(p_vec), 1)
            p_vec = torch.tensor(p_vec)
            revenue = reven_array_true[i]
            sum = sum + ((torch.transpose(mu+sigma, 0, 1) @
                         p_vec - torch.tensor([[revenue]]))**2)/t
        # print(sum.size())
        external_grad = torch.tensor([[1]])
        sum.backward(gradient=external_grad)
        mu_ = (mu - alpha*mu.grad).detach().numpy()
        sigma_ = (sigma - alpha*sigma.grad).detach().numpy()
        w = mu_ + sigma_ * np.random.normal(0, 1)
    #     print(w)
        price_estimate = minimize(g_fun, x0=np.array(
            [random.random()]), args=(w,), bounds=[(pl, ph)])["x"][0]
        p_array.append(price_estimate)
        p_curr = p_vector(price_estimate)
        # print(p_curr)
        # epsilon = max(0,100-reven_array_true[-1])
        epsilon = np.max(
            [0, np.min(optimal_reward - np.array(reven_array_true))])
        # error += epsilon
        epsilon_array.append(epsilon)
    return np.array(epsilon_array), reven_array_true

In [ ]:
best_till_now_PS = []
reven_PS = []
for i in range(10):
    epsilon, rets = ParameterSpace(i % 10)
    reven_PS.append(rets)
    # cum_regret_PS.append(ParameterSpace())
    best_till_now_PS.append(epsilon)

target_PS = np.mean(reven_PS, axis=0)

target_PS